<a href="https://colab.research.google.com/github/n-schiozer/met_comp_economistas_unesp/blob/main/Aula_08.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 8 — Pandas: agregação, transformação, filtragem e operações condicionais


**Objetivos:** (1) calcular estatísticas e agrupamentos; (2) criar indicadores e participações percentuais; (3) filtrar observações; (4) criar classificações e calcular agregações condicionais.

### Referências

**Bibliografia principal:**

McKINNEY, Wes. *Python for Data Analysis: Data Wrangling with pandas, NumPy, and Jupyter*. 3. ed. Sebastopol: O'Reilly Media, 2022.

- **Capítulo 5 — Getting Started with pandas:** seleção, filtragem, ordenação e manipulação de DataFrames.
- **Capítulo 10 — Data Aggregation and Group Operations:** `groupby()`, `agg()`, `transform()` e operações de agregação.

**Livro disponível gratuitamente:** [Python for Data Analysis — 3ª edição](https://wesmckinney.com/book/).

**Fonte dos dados:**

- [World Bank — World Development Indicators (WDI)](https://databank.worldbank.org/source/world-development-indicators).
- [World Bank — Documentação da API](https://datahelpdesk.worldbank.org/knowledgebase/articles/889392).

**Documentação complementar:**

- [Pandas — GroupBy](https://pandas.pydata.org/docs/user_guide/groupby.html).
- [Pandas — Indexing and Selecting Data](https://pandas.pydata.org/docs/user_guide/indexing.html).


## Preparação — obter a base do Banco Mundial

Primeiro executaremos o código fornecido para construir `dados` com sete colunas: `ano`, `pais`, `regiao`, `pib_usd`, `populacao_total`, `inflacao_anual_pct` e `desemprego_pct`.

- **PIB:** valor nominal em dólares correntes (US$), não ajustado por poder de compra.
- **Inflação:** variação anual do índice de preços ao consumidor (%).
- **Desemprego:** percentual da força de trabalho; série de estimativas modeladas da OIT no WDI.
- **Região:** classificação utilizada pelo Banco Mundial.

**Atenção:** os dados podem ter valores ausentes (`NaN`)

In [ ]:
import requests                # Biblioteca para consultar APIs e páginas na internet.
import pandas as pd            # Biblioteca para manipular dados em DataFrames.

# ============================================================
# 1. DEFINIR PERÍODO E INDICADORES
# ============================================================

ano_inicial = 2020
ano_final = 2024

# Chave: nome que desejamos no DataFrame. Valor: código da API.
# Fonte: https://databank.worldbank.org/source/world-development-indicators
indicadores = {
    "pib_usd": "NY.GDP.MKTP.CD",            # PIB nominal em dólares.
    "populacao_total": "SP.POP.TOTL",       # População total.
    "inflacao_anual_pct": "FP.CPI.TOTL.ZG", # Inflação anual (%).
    "desemprego_pct": "SL.UEM.TOTL.ZS",     # Desemprego (% da força de trabalho).
}

In [ ]:
# ============================================================
# 2. OBTER INFORMAÇÕES SOBRE OS PAÍSES
# ============================================================

url_paises = "https://api.worldbank.org/v2/country"

# Solicitar dados no formato JSON e até 400 países/economias na página.
resposta = requests.get(
    url_paises,
    params={"format": "json", "per_page": 400},
    timeout=45
)
resposta.raise_for_status()  # Interrompe a execução se houver erro HTTP.
conteudo = resposta.json()   # Transforma o JSON em estruturas do Python.

# Construir uma lista de dicionários: um registro por país/economia.
# A expressão for ... if ... seleciona apenas países, excluindo agregados.
metadados = pd.DataFrame([
    {
        "codigo_pais": pais["id"],
        "pais": pais["name"],
        "regiao": pais["region"]["value"]
    }
    for pais in conteudo[1]
    if pais["region"]["value"] != "Aggregates"
])

# set() cria um conjunto de códigos únicos para verificar quais são válidos.
codigos_validos = set(metadados["codigo_pais"])

In [ ]:
# ============================================================
# 3. BAIXAR OS INDICADORES ECONÔMICOS
# ============================================================

registros = []  # Lista vazia para guardar todas as observações.

# .items() percorre cada par nome-do-indicador/código-da-API.
for coluna, codigo in indicadores.items():

    # f-string: substitui {codigo} pelo código atual do indicador.
    url = f"https://api.worldbank.org/v2/country/all/indicator/{codigo}"

    resposta = requests.get(
        url,
        params={
            "format": "json",                    # Resposta JSON.
            "source": 2,                         # World Development Indicators.
            "date": f"{ano_inicial}:{ano_final}", # Intervalo de anos.
            "per_page": 20000                    # Limite de registros por página.
        },
        timeout=45
    )
    resposta.raise_for_status()  # Verificar se o pedido foi bem-sucedido.
    conteudo = resposta.json()

    # Cada item representa um país, um ano e um indicador.
    for item in conteudo[1]:
        if item["countryiso3code"] in codigos_validos:
            registros.append({
                "codigo_pais": item["countryiso3code"],
                "ano": int(item["date"]),        # Ano convertido para inteiro.
                "indicador": coluna,              # Nome da variável.
                "valor": item["value"]           # Valor ou None (ausente).
            })

In [ ]:
# ============================================================
# 4. ORGANIZAR OS DADOS EM TABELA
# ============================================================

dados_longos = pd.DataFrame(registros)

# No formato longo, cada linha identifica país, ano, indicador e valor.
# pivot() coloca cada indicador em uma coluna; não agrega estatísticas.
dados_largos = (
    dados_longos
    .pivot(index=["codigo_pais", "ano"],
           columns="indicador", values="valor")
    .reset_index()  # Traz código e ano do índice de volta às colunas.
)

# Elimina somente o nome do eixo das colunas, sem apagar as colunas.
dados_largos.columns.name = None

In [ ]:
# ============================================================
# 5. JUNTAR METADADOS E ORGANIZAR O DATAFRAME FINAL
# ============================================================

dados = (
    dados_largos
    # merge(): juntar indicadores e metadados usando o código do país.
    .merge(metadados, on="codigo_pais", how="inner",
           validate="many_to_one")
    # Manter e ordenar as sete colunas de interesse.
    [["ano", "pais", "regiao", "pib_usd", "populacao_total",
      "inflacao_anual_pct", "desemprego_pct"]]
    .sort_values(["ano", "pais"])  # Ordenar por ano e país.
    .reset_index(drop=True)        # Reiniciar o índice sem preservar o antigo.
)

print("Dimensões:", dados.shape)
print("Países:", dados["pais"].nunique())
print("Valores ausentes por coluna:")
print(dados.isna().sum())

dados.head(10)

Dimensões: (1085, 7)
Países: 217
Valores ausentes por coluna:
ano                     0
pais                    0
regiao                  0
pib_usd                52
populacao_total         0
inflacao_anual_pct    195
desemprego_pct        159
dtype: int64


,ano,pais,regiao,pib_usd,populacao_total,inflacao_anual_pct,desemprego_pct
0,2020,Afghanistan,"Middle East, North Africa, Afghanistan & Pakistan",1.995593e+10,39068979.0,5.601888,11.710
1,2020,Albania,Europe & Central Asia,1.524146e+10,2528480.0,1.620887,11.639
2,2020,Algeria,"Middle East, North Africa, Afghanistan & Pakistan",1.648734e+11,44042091.0,2.415131,14.080
3,2020,American Samoa,East Asia & Pacific,7.210000e+08,49761.0,NaN,NaN
4,2020,Andorra,Europe & Central Asia,2.891001e+09,77380.0,NaN,NaN
5,2020,Angola,Sub-Saharan Africa,5.851203e+10,33451132.0,22.271539,16.696
6,2020,Antigua and Barbuda,Latin America & Caribbean,1.411637e+09,91846.0,0.625990,NaN
7,2020,Argentina,Latin America & Caribbean,3.857405e+11,45191965.0,42.015095,11.461
8,2020,Armenia,Europe & Central Asia,1.264170e+10,2961500.0,1.211436,18.175
9,2020,Aruba,Latin America & Caribbean,2.461056e+09,108587.0,NaN,NaN


---
## 1. Agregação: calcular estatísticas resumidas

Uma **agregação** resume várias observações em um número: `sum()` (total), `mean()` (média), `count()` (quantidade **não ausente**) e `size()` (quantidade de linhas, mesmo com valores ausentes).

**Regra importante:** cada país aparece em cinco anos. Para somar PIB ou população em um ano, devemos antes escolher **qual ano**; somar o PIB de 2020 a 2024 não representa o PIB de um único período.

In [ ]:
# Selecionar somente o ano de 2024 para não somar anos diferentes.
dados_2024 = dados.loc[dados["ano"] == 2024].copy()

print(dados_2024) # Estão organizados em ordem alfabética



       ano                   pais  \
868   2024            Afghanistan   
869   2024                Albania   
870   2024                Algeria   
871   2024         American Samoa   
872   2024                Andorra   
...    ...                    ...   
1080  2024  Virgin Islands (U.S.)   
1081  2024     West Bank and Gaza   
1082  2024            Yemen, Rep.   
1083  2024                 Zambia   
1084  2024               Zimbabwe   

                                                 regiao       pib_usd  \
868   Middle East, North Africa, Afghanistan & Pakistan  1.777851e+10   
869                               Europe & Central Asia  2.703747e+10   
870   Middle East, North Africa, Afghanistan & Pakistan  2.693223e+11   
871                                 East Asia & Pacific           NaN   
872                               Europe & Central Asia  4.044250e+09   
...                                                 ...           ...   
1080                         Latin America &

In [ ]:
print("PIB total dos países com informação em 2024 (US$):",
      dados_2024["pib_usd"].sum())

print("PIB médio por país com informação (US$):",
      dados_2024["pib_usd"].mean())

print("Países/linhas em 2024:", dados_2024.shape[0])

print("Países com inflação informada:",
      dados_2024["inflacao_anual_pct"].count())

print("Média simples da inflação entre países com dado (%):",
      dados_2024["inflacao_anual_pct"].mean())

PIB total dos países com informação em 2024 (US$): 110620750372595.44
PIB médio por país com informação (US$): 553103751862.9772
Países/linhas em 2024: 217
Países com inflação informada: 176
Média simples da inflação entre países com dado (%): 7.407924533441153


### 1. sort_values() — apenas ordena as linhas

Aqui você continua tendo um DataFrame, mas com os países organizados por região. É possível visualizar normalmente.

In [ ]:
ordenados = dados_2024.sort_values("regiao")

display(ordenados)

,ano,pais,regiao,pib_usd,populacao_total,inflacao_anual_pct,desemprego_pct
976,2024,Lao PDR,East Asia & Pacific,1.650293e+10,7769819.0,23.130570,1.174
1031,2024,Samoa,East Asia & Pacific,1.175750e+09,218019.0,2.172455,4.855
934,2024,Fiji,East Asia & Pacific,5.968126e+09,928784.0,3.890842,5.361
1069,2024,Tuvalu,East Asia & Pacific,5.675227e+07,9646.0,NaN,NaN
972,2024,"Korea, Rep.",East Asia & Pacific,1.875388e+12,51751065.0,2.321743,2.784
...,...,...,...,...,...,...,...
939,2024,"Gambia, The",Sub-Saharan Africa,2.404889e+09,2759988.0,11.561053,6.480
938,2024,Gabon,Sub-Saharan Africa,2.089568e+10,2538952.0,1.173120,20.091
1053,2024,Sudan,Sub-Saharan Africa,4.967244e+10,50448963.0,NaN,NaN
931,2024,Eswatini,Sub-Saharan Africa,4.858886e+09,1242822.0,NaN,34.643


## 1.1 Um agrupamento: comparar regiões

`groupby("regiao")` separa as observações em grupos. Em seguida, usamos `sum()` para obter o PIB nominal agregado dos países **com dados disponíveis** em cada região.

Não é necessário conhecer de antemão a lista de regiões.

### Etapa 1 — Agrupar os dados por região

O método groupby() permite dividir as observações de um DataFrame em grupos, de acordo com os valores de uma ou mais colunas.

Neste exemplo, utilizamos a coluna regiao para agrupar os países. Assim, cada grupo reúne todos os países pertencentes à mesma região geográfica.

É importante observar que groupby() não calcula nenhuma estatística e não produz imediatamente uma nova tabela. Ele cria um objeto de agrupamento (DataFrameGroupBy), sobre o qual podemos aplicar operações como soma (sum()), média (mean()) e contagem (size()).

Podemos também examinar os grupos antes de realizar qualquer cálculo, utilizando get_group() para visualizar os países de determinada região.


In [ ]:
agrupamento = dados_2024.groupby("regiao")

print(agrupamento)


In [ ]:
print(list(agrupamento.groups.keys()))

print(agrupamento.size())

print( agrupamento.get_group("Latin America & Caribbean ")) # CUIDADO com o espaço

['East Asia & Pacific', 'Europe & Central Asia', 'Latin America & Caribbean ', 'Middle East, North Africa, Afghanistan & Pakistan', 'North America', 'South Asia', 'Sub-Saharan Africa ']
regiao
East Asia & Pacific                                  37
Europe & Central Asia                                58
Latin America & Caribbean                            42
Middle East, North Africa, Afghanistan & Pakistan    23
North America                                         3
South Asia                                            6
Sub-Saharan Africa                                   48
dtype: int64
       ano                            pais                      regiao  \
874   2024             Antigua and Barbuda  Latin America & Caribbean    
875   2024                       Argentina  Latin America & Caribbean    
877   2024                           Aruba  Latin America & Caribbean    
881   2024                    Bahamas, The  Latin America & Caribbean    
884   2024                      

## Etapa 2 — Calcular o PIB total e o PIB per capita médio

O método .agg() (abreviação de aggregation, ou agregação) permite calcular uma ou mais estatísticas sobre os dados, como soma, média, mínimo, máximo e contagem.

Quando utilizado após groupby(), o .agg() calcula essas estatísticas separadamente para cada grupo.

Uma vantagem do .agg() é permitir que diferentes operações sejam aplicadas a diferentes colunas, produzindo uma tabela com os resultados.

No nosso exemplo, utilizamos:

pib_total=("pib_usd", "sum"): cria uma coluna chamada pib_total, somando os PIBs dos países de cada região.

pib_per_capita_medio=("pib_per_capita", "mean"): cria uma coluna chamada pib_per_capita_medio, calculando a média do PIB per capita dos países de cada região.

A estrutura utilizada é:

nome_nova_coluna=("coluna_original", "operação")

Assim, combinamos groupby() para definir os grupos e .agg() para determinar quais estatísticas serão calculadas em cada grupo.

In [ ]:
# 1. Calcular o PIB per capita primeiro para que a coluna exista
dados_2024["pib_per_capita"] = dados_2024["pib_usd"] / dados_2024["populacao_total"]



,pib_total,pib_per_capita_medio
regiao,,
East Asia & Pacific,3.136614e+13,19083.921705
Europe & Central Asia,2.934445e+13,43355.751279
Latin America & Caribbean,6.916569e+12,18491.763437
"Middle East, North Africa, Afghanistan & Pakistan",5.073409e+12,18854.228395
North America,3.157728e+13,94478.601348
South Asia,4.364256e+12,4799.742101
Sub-Saharan Africa,1.978644e+12,2681.581896


In [ ]:
# 2. Agora o agrupamento e agregação funcionarão sem erros
pib_por_regiao = agrupamento.agg(
    pib_total=("pib_usd", "sum"),
    pib_per_capita_medio=("pib_per_capita", "mean")
)

pib_por_regiao

## Pergunta: como conseguimos rodar o código anterior sendo que "agrupamento" foi criado antes de dados_2024["pib_per_capita"]?



<details>
<summary><b>Por que o agrupamento reconhece uma coluna criada depois?</b></summary>

Isso funciona porque o `groupby()` não cria uma cópia independente do DataFrame original. Ele mantém uma referência ao DataFrame utilizado para formar os grupos.

</details>

In [ ]:
# Resultado final:

pib_por_regiao

,pib_total,pib_per_capita_medio
regiao,,
East Asia & Pacific,3.136614e+13,19083.921705
Europe & Central Asia,2.934445e+13,43355.751279
Latin America & Caribbean,6.916569e+12,18491.763437
"Middle East, North Africa, Afghanistan & Pakistan",5.073409e+12,18854.228395
North America,3.157728e+13,94478.601348
South Asia,4.364256e+12,4799.742101
Sub-Saharan Africa,1.978644e+12,2681.581896


### 1.2 Várias estatísticas de uma vez com `agg()`

Podemos escolher **estatísticas diferentes para colunas diferentes**. A média de inflação e desemprego neste exemplo é uma **média simples entre países com dados disponíveis**, não a inflação ou o desemprego agregado da região.

In [ ]:
resumo_regional = (
    dados_2024.groupby("regiao")
    .agg(
        pib_total_usd=("pib_usd", "sum"),
        populacao_total=("populacao_total", "sum"),
        inflacao_media_pct=("inflacao_anual_pct", "mean"),
        paises_com_inflacao=("inflacao_anual_pct", "count")
    )
)

resumo_regional

,pib_total_usd,populacao_total,inflacao_media_pct,paises_com_inflacao
regiao,,,,
East Asia & Pacific,3.136614e+13,2.364907e+09,3.046839,26
Europe & Central Asia,2.934445e+13,9.285669e+08,4.317827,49
Latin America & Caribbean,6.916569e+12,6.621856e+08,11.463234,30
"Middle East, North Africa, Afghanistan & Pakistan",5.073409e+12,8.131739e+08,9.518263,21
North America,3.157728e+13,3.813308e+08,2.665555,2
South Asia,4.364256e+12,1.677385e+09,3.974462,6
Sub-Saharan Africa,1.978644e+12,1.289936e+09,10.477261,42


### 1.3 Agrupamento por duas colunas: região **e** ano

O grupo agora é definido pelo par `("ano", "regiao")`. Isso permite comparar regiões ao longo do tempo.

In [ ]:
regiao_ano = (
    dados.groupby(["ano", "regiao"])
    .agg(
        pib_total_usd=("pib_usd", "sum"),
        inflacao_media_pct=("inflacao_anual_pct", "mean"),
        paises_com_inflacao=("inflacao_anual_pct", "count")
    )
)

regiao_ano.head(15)

pib_total_usd  \
ano  regiao                                                             
2020 East Asia & Pacific                                 2.701904e+13   
     Europe & Central Asia                               2.234080e+13   
     Latin America & Caribbean                           4.626281e+12   
     Middle East, North Africa, Afghanistan & Pakistan   3.572057e+12   
     North America                                       2.303786e+13   
     South Asia                                          3.172770e+12   
     Sub-Saharan Africa                                  1.894759e+12   
2021 East Asia & Pacific                                 3.106643e+13   
     Europe & Central Asia                               2.560614e+13   
     Latin America & Caribbean                           5.257529e+12   
     Middle East, North Africa, Afghanistan & Pakistan   4.296079e+12   
     North America                                       2.575531e+13   
     South Asia                                          3.717045e+12   
     Sub-Saharan Africa                                  2.131948e+12   
2022 East Asia & Pacific                                 3.066873e+13   

                                                        inflacao_media_pct  \
ano  regiao                                                                  
2020 East Asia & Pacific                                          1.444751   
     Europe & Central Asia                                        2.031667   
     Latin America & Caribbean                                    4.774045   
     Middle East, North Africa, Afghanistan & Pakistan            6.952704   
     North America                                                0.975292   
     South Asia                                                   4.630734   
     Sub-Saharan Africa                                          22.416046   
2021 East Asia & Pacific                                          2.568007   
     Europe & Central Asia                                        4.473458   
     Latin America & Caribbean                                    7.401867   
     Middle East, North Africa, Afghanistan & Pakistan           12.289963   
     North America                                                4.046526   
     South Asia                                                   4.951429   
     Sub-Saharan Africa                                          16.685970   
2022 East Asia & Pacific                                          6.582769   

                                                        paises_com_inflacao  
ano  regiao                                                                  
2020 East Asia & Pacific                                                 27  
     Europe & Central Asia                                               49  
     Latin America & Caribbean                                           30  
     Middle East, North Africa, Afghanistan & Pakistan                   21  
     North America                                                        2  
     South Asia                                                           6  
     Sub-Saharan Africa                                                  44  
2021 East Asia & Pacific                                                 27  
     Europe & Central Asia                                               49  
     Latin America & Caribbean                                           30  
     Middle East, North Africa, Afghanistan & Pakistan                   21  
     North America                                                        2  
     South Asia                                                           6  
     Sub-Saharan Africa                                                  44  
2022 East Asia & Pacific                                                 27

#### Exercício 1 — agregações

1. Calcule a **população total em 2024**, somando apenas os países com informação.
2. Calcule o **desemprego médio por região em 2024** e ordene da maior para a menor média.
3. Crie uma tabela por **ano e região** com: população total, inflação média e número de observações com desemprego informado.


In [ ]:
# EXERCÍCIO 1 — complete aqui.
# Dicas: dados_2024, groupby(...,) e agg(...).

---
## 2. Transformação: criar variáveis novas

Uma **transformação** cria uma coluna para cada linha, preservando o conjunto de observações. Podemos dividir, multiplicar ou combinar colunas existentes.

Por exemplo, o **PIB nominal per capita (US$/pessoa)** é calculado como PIB nominal dividido pela população.

In [ ]:
# Novas colunas calculadas diretamente a partir das existentes.
dados["pib_bilhoes_usd"] = dados["pib_usd"] / 1_000_000_000
dados["populacao_milhoes"] = dados["populacao_total"] / 1_000_000

dados["pib_per_capita_usd"] = (
    dados["pib_usd"] / dados["populacao_total"]
)

dados[["ano", "pais", "pib_bilhoes_usd",
       "populacao_milhoes", "pib_per_capita_usd"]].head(10)

,ano,pais,pib_bilhoes_usd,populacao_milhoes,pib_per_capita_usd
0,2020,Afghanistan,19.955929,39.068979,510.787063
1,2020,Albania,15.241459,2.528480,6027.913507
2,2020,Algeria,164.873415,44.042091,3743.541952
3,2020,American Samoa,0.721000,0.049761,14489.258656
4,2020,Andorra,2.891001,0.077380,37361.090067
5,2020,Angola,58.512034,33.451132,1749.179484
6,2020,Antigua and Barbuda,1.411637,0.091846,15369.608225
7,2020,Argentina,385.740508,45.191965,8535.599380
8,2020,Armenia,12.641699,2.961500,4268.680933
9,2020,Aruba,2.461056,0.108587,22664.370995


### 2.1 Transformação em várias etapas: participação no PIB da região

Queremos calcular, para cada país e ano:

$$
\text{Participação no PIB da região (%)} =
100 \times \frac{\text{PIB do país}}{\text{PIB dos países da região no mesmo ano}}
$$

`groupby(...).transform("sum")` calcula o total de cada grupo **e o repete nas linhas originais**, permitindo dividir o PIB de cada país pelo total do seu grupo.

**Atenção:** este denominador inclui apenas os países com PIB disponível na base; não é necessariamente idêntico a uma estatística regional oficial do Banco Mundial.

In [ ]:
# Para cada país, repetir o PIB total de sua região e de seu ano.
dados["pib_total_regiao_ano"] = (
    dados.groupby(["ano", "regiao"])["pib_usd"].transform("sum") # groupby (agrupamento) + transform("sum") : faz a soma por agrupamento
)

dados[dados["ano"] == 2022].sort_values("regiao").head(10)

,ano,pais,regiao,pib_usd,populacao_total,inflacao_anual_pct,desemprego_pct,pib_total_regiao_ano,participacao_pib_regional_pct,pib_bilhoes_usd,populacao_milhoes,pib_per_capita_usd
542,2022,Lao PDR,East Asia & Pacific,1.546879e+10,7.559007e+06,22.956223,1.209,3.066873e+13,0.050438,15.468785,7.559007,2046.404403
597,2022,Samoa,East Asia & Pacific,8.895547e+08,2.152610e+05,10.961882,5.048,3.066873e+13,0.002901,0.889555,0.215261,4132.447177
500,2022,Fiji,East Asia & Pacific,4.977862e+09,9.194220e+05,6.078824,5.403,3.066873e+13,0.016231,4.977862,0.919422,5414.120921
635,2022,Tuvalu,East Asia & Pacific,5.410415e+07,9.992000e+03,NaN,NaN,3.066873e+13,0.000176,0.054104,0.009992,5414.746429
538,2022,"Korea, Rep.",East Asia & Pacific,1.799363e+12,5.167257e+07,5.089514,2.857,3.066873e+13,5.867093,1799.363117,51.672569,34822.404840
537,2022,"Korea, Dem. People's Rep.",East Asia & Pacific,NaN,2.632884e+07,NaN,3.457,3.066873e+13,NaN,NaN,26.328845,NaN
536,2022,Kiribati,East Asia & Pacific,2.712144e+08,1.304690e+05,5.345954,NaN,3.066873e+13,0.000884,0.271214,0.130469,2078.764875
475,2022,China,East Asia & Pacific,1.831677e+13,1.412175e+09,1.973576,4.980,3.066873e+13,59.724560,18316.765022,1412.175000,12970.605641
605,2022,Singapore,East Asia & Pacific,5.142525e+11,5.637022e+06,6.130954,3.591,3.066873e+13,1.676798,514.252535,5.637022,91227.697043
462,2022,Brunei Darussalam,East Asia & Pacific,1.668154e+10,4.553700e+05,3.682250,5.191,3.066873e+13,0.054393,16.681536,0.455370,36632.928095


In [ ]:
# Dividir o PIB de cada país pelo total de seu grupo.
dados["participacao_pib_regional_pct"] = (
    100 * dados["pib_usd"] / dados["pib_total_regiao_ano"]
)

dados[["ano", "pais", "regiao", "pib_bilhoes_usd",
       "participacao_pib_regional_pct"]].head(10)

,ano,pais,regiao,pib_bilhoes_usd,participacao_pib_regional_pct
0,2020,Afghanistan,"Middle East, North Africa, Afghanistan & Pakistan",19.955929,0.558668
1,2020,Albania,Europe & Central Asia,15.241459,0.068223
2,2020,Algeria,"Middle East, North Africa, Afghanistan & Pakistan",164.873415,4.615643
3,2020,American Samoa,East Asia & Pacific,0.721000,0.002668
4,2020,Andorra,Europe & Central Asia,2.891001,0.012940
5,2020,Angola,Sub-Saharan Africa,58.512034,3.088098
6,2020,Antigua and Barbuda,Latin America & Caribbean,1.411637,0.030513
7,2020,Argentina,Latin America & Caribbean,385.740508,8.338026
8,2020,Armenia,Europe & Central Asia,12.641699,0.056586
9,2020,Aruba,Latin America & Caribbean,2.461056,0.053197


### 2.2 Verificação: as participações somam 100%?

Como as participações são calculadas dentro de cada par **ano/região**, espera-se uma soma aproximadamente igual a 100% quando existe PIB positivo no grupo. Grupos sem valores válidos podem produzir `NaN`.

In [ ]:
verificacao = (
    dados.groupby(["ano", "regiao"])["participacao_pib_regional_pct"]
    .sum()  # Mantém NaN se nenhuma participação for observada.
)

verificacao.round(2).head(15)

ano   regiao                                           
2020  East Asia & Pacific                                  100.0
      Europe & Central Asia                                100.0
      Latin America & Caribbean                            100.0
      Middle East, North Africa, Afghanistan & Pakistan    100.0
      North America                                        100.0
      South Asia                                           100.0
      Sub-Saharan Africa                                   100.0
2021  East Asia & Pacific                                  100.0
      Europe & Central Asia                                100.0
      Latin America & Caribbean                            100.0
      Middle East, North Africa, Afghanistan & Pakistan    100.0
      North America                                        100.0
      South Asia                                           100.0
      Sub-Saharan Africa                                   100.0
2022  East Asia & Pacific                                  100.0
Name: participacao_pib_regional_pct, dtype: float64

# Para mostrar os 5 países com maior participação no PIB de cada região em 2024

podemos combinar sort_values() e groupby().

In [ ]:
# 1. Selecionar os dados de 2024
dados_2024 = dados.loc[dados["ano"] == 2024]

# 2. Ordenar os países pela participação no PIB regional
dados_ordenados = dados_2024.sort_values(
    ["regiao", "participacao_pib_regional_pct"],
    ascending=[True, False]
)

# O parâmetro ascending do Pandas define a ordem de classificação dos valores no método sort_values().

# 3. Selecionar os 5 maiores países de cada região
maiores_participacoes = (
    dados_ordenados.groupby("regiao")
    .head(5)
)

# 4. Selecionar as colunas e exibir a tabela
display(maiores_participacoes[[
    "pais", "regiao", "pib_bilhoes_usd",
    "participacao_pib_regional_pct"
]])

,pais,regiao,pib_bilhoes_usd,participacao_pib_regional_pct
909,China,East Asia & Pacific,18729.668436,59.713021
966,Japan,East Asia & Pacific,4190.008188,13.358381
972,"Korea, Rep.",East Asia & Pacific,1875.388209,5.979022
878,Australia,East Asia & Pacific,1757.022452,5.601654
958,Indonesia,East Asia & Pacific,1396.301788,4.451622
941,Germany,Europe & Central Asia,4685.592578,15.967559
1073,United Kingdom,Europe & Central Asia,3695.539514,12.593657
936,France,Europe & Central Asia,3160.442622,10.770154
964,Italy,Europe & Central Asia,2383.435562,8.122270
1029,Russian Federation,Europe & Central Asia,2186.462269,7.451025


#### Exercício 2 — transformações

1. Crie `populacao_regiao_ano` com `groupby(...).transform("sum")`, repetindo em cada linha a população total da sua região no mesmo ano.
2. Crie `participacao_pop_regional_pct` como **100 × população do país / população da região no mesmo ano**.
3. Verifique se a soma das participações de cada par **ano/região** é aproximadamente 100% (quando houver dados).
4. **Interprete:** por que não devemos somar os valores de `pib_per_capita_usd` para obter o PIB per capita regional?

In [ ]:
# EXERCÍCIO 2 — complete aqui.
# Dica: use como modelo o cálculo de participação no PIB regional.

---
## 3. Filtragem: escolher observações

**Filtrar** significa selecionar linhas que atendem a uma condição, sem modificar a tabela original. A sintaxe básica é `dados.loc[condicao, colunas]`.

Vamos praticar: condição simples; várias condições com `&` (**E**) ou `|` (**OU**); negação `~`; valores ausentes; e busca em texto.

In [ ]:
# Selecionar países observados em 2024.
observacoes_2024 = dados.loc[dados["ano"] == 2024]

observacoes_2024[["ano", "pais", "pib_per_capita_usd"]].head(10)

,ano,pais,pib_per_capita_usd
868,2024,Afghanistan,416.871146
869,2024,Albania,11374.008578
870,2024,Algeria,5752.990767
871,2024,American Samoa,NaN
872,2024,Andorra,49357.437070
873,2024,Angola,2720.819007
874,2024,Antigua and Barbuda,23059.833070
875,2024,Argentina,13969.783660
876,2024,Armenia,8556.214070
877,2024,Aruba,38590.565029


In [ ]:
# Países com inflação >= 10% em 2024.
inflacao_alta_2024 = dados.loc[
    (dados["ano"] == 2024) & (dados["inflacao_anual_pct"] >= 10),
    ["pais", "regiao", "inflacao_anual_pct"]
]


print("Inflação >= 10% em 2024:") #

print(inflacao_alta_2024.head(4)) # print() exibe o DataFrame como texto
display(inflacao_alta_2024.head(4)) # com display() Isso exibirá uma tabela formatada, com as colunas lado a lado.

Inflação >= 10% em 2024:
           pais                      regiao  inflacao_anual_pct
873      Angola         Sub-Saharan Africa            28.240495
875   Argentina  Latin America & Caribbean           219.883929
883  Bangladesh                  South Asia           10.465748
899     Burundi         Sub-Saharan Africa            20.212493


,pais,regiao,inflacao_anual_pct
873,Angola,Sub-Saharan Africa,28.240495
875,Argentina,Latin America & Caribbean,219.883929
883,Bangladesh,South Asia,10.465748
899,Burundi,Sub-Saharan Africa,20.212493


### 3.1 Múltiplas condições e listas de valores

Use parênteses em **cada comparação** ao combinar condições. `.isin()` permite procurar vários valores de uma só vez.

In [ ]:

# Brasil, México ou Argentina, em 2023 ou 2024.
selecionados = dados.loc[
    (dados["ano"].isin([2023, 2024])) &
    (dados["pais"].isin(["Brazil", "Mexico", "Argentina"]))
]


print("Países escolhidos:")
display(selecionados[["ano", "pais", "inflacao_anual_pct"]])

Países escolhidos:


,ano,pais,inflacao_anual_pct
658,2023,Argentina,133.488936
677,2023,Brazil,4.593563
778,2023,Mexico,5.527961
875,2024,Argentina,219.883929
894,2024,Brazil,4.367464
995,2024,Mexico,4.722256


In [ ]:
# Reorganizar o índice

paises_escolhidos = selecionados.reset_index(drop=True)

display(paises_escolhidos[["ano", "pais", "inflacao_anual_pct"]])

,ano,pais,inflacao_anual_pct
0,2023,Argentina,133.488936
1,2023,Brazil,4.593563
2,2023,Mexico,5.527961
3,2024,Argentina,219.883929
4,2024,Brazil,4.367464
5,2024,Mexico,4.722256


### 3.2 Valores ausentes e padrões de texto

- `.isna()` identifica registros sem informação; `.notna()` identifica informações presentes.
- `.str.contains("Latin America", case=False, na=False)` verifica se um **texto** contém determinada expressão, independentemente de maiúsculas/minúsculas.

Esse recurso também evita depender de escrever o nome completo da região.

In [ ]:
# Exemplos de inflação ausente em 2024.
sem_inflacao_2024 = dados.loc[
    (dados["ano"] == 2024) & (dados["inflacao_anual_pct"].isna())
]


print("Exemplos de inflação ausente:")
display(sem_inflacao_2024.head(10))


Exemplos de inflação ausente:


,ano,pais,regiao,pib_usd,populacao_total,inflacao_anual_pct,desemprego_pct,pib_total_regiao_ano,participacao_pib_regional_pct,pib_bilhoes_usd,populacao_milhoes,pib_per_capita_usd
871,2024,American Samoa,East Asia & Pacific,NaN,46765.0,NaN,NaN,3.136614e+13,NaN,NaN,0.046765,NaN
872,2024,Andorra,Europe & Central Asia,4.044250e+09,81938.0,NaN,NaN,2.934445e+13,0.013782,4.044250,0.081938,49357.437070
877,2024,Aruba,Latin America & Caribbean,4.167588e+09,107995.0,NaN,NaN,6.916569e+12,0.060255,4.167588,0.107995,38590.565029
889,2024,Bermuda,North America,9.194499e+09,64636.0,NaN,NaN,3.157728e+13,0.029117,9.194499,0.064636,142250.433195
895,2024,British Virgin Islands,Latin America & Caribbean,NaN,39471.0,NaN,NaN,6.916569e+12,NaN,NaN,0.039471,NaN
904,2024,Cayman Islands,Latin America & Caribbean,7.765337e+09,74457.0,NaN,NaN,6.916569e+12,0.112272,7.765337,0.074457,104292.900672
907,2024,Channel Islands,Europe & Central Asia,NaN,168126.0,NaN,6.031,2.934445e+13,NaN,NaN,0.168126,NaN
912,2024,"Congo, Dem. Rep.",Sub-Saharan Africa,7.571618e+10,109276265.0,NaN,4.421,1.978644e+12,3.826670,75.716176,109.276265,692.887665
917,2024,Cuba,Latin America & Caribbean,NaN,10979783.0,NaN,1.755,6.916569e+12,NaN,NaN,10.979783,NaN
918,2024,Curacao,Latin America & Caribbean,3.561178e+09,155967.0,NaN,NaN,6.916569e+12,0.051488,3.561178,0.155967,22832.895499


### 3.3 Condições nomeadas e o oposto de uma seleção

Uma condição pode ser armazenada em uma variável booleana, com valores `True` e `False`. O operador `~` inverte uma condição: seleciona o **contrário** dela.

In [ ]:
ano_recente = dados["ano"] >= 2023
inflacao_alta = dados["inflacao_anual_pct"] >= 10
inflacao_informada = dados["inflacao_anual_pct"].notna()

# Condições combinadas: ano >= 2023, inflação >= 10% e dado presente.
selecao = ano_recente & inflacao_alta & inflacao_informada

display(selecao)



,0
0,False
1,False
2,False
3,False
4,False
...,...
1080,False
1081,True
1082,False
1083,True


In [ ]:
print("Selecionados:")
display(dados.loc[selecao, ["ano", "pais", "inflacao_anual_pct"]].head(10))

# O contrário: casos que NÃO satisfazem todas essas condições.
print("Fora da seleção:")
display(dados.loc[~selecao, ["ano", "pais", "inflacao_anual_pct"]].head(10))

#### Exercício 3 — filtragem

1. Selecione países da **América Latina em 2024** com desemprego informado.
2. Selecione países de **2024** com inflação **maior que 8% OU** desemprego **maior que 12%**.
3. Selecione observações de **2024 com PIB ausente** (`isna()`) e mostre dez linhas.
4. Crie uma condição para os anos 2020 e 2021, selecione essas linhas e depois selecione o **complemento** usando `~`.

Para cada filtro, use `.shape[0]` para verificar quantas observações foram encontradas.

In [ ]:
# EXERCÍCIO 3 — complete aqui.
# Dicas: .loc[], &, |, ~, .isin(), .isna(), .notna(), .str.contains().

---
## 4. Operações condicionais: cálculos e classificações

Enquanto um filtro **retira linhas**, uma transformação condicional permite **manter todas as linhas**, atribuindo valores diferentes a uma coluna conforme condições.

Para não confundir informações ausentes com inflação baixa, vamos criar **três categorias**.

In [ ]:
# Criar uma classificação inicial para todas as observações.
dados["faixa_inflacao"] = "Sem informação"

# Se a inflação estiver informada, a categoria passa a ser "Abaixo de 10%".
dados.loc[dados["inflacao_anual_pct"].notna(), "faixa_inflacao"] = "Abaixo de 10%"

# Substituir pelas observações com inflação de 10% ou mais.
dados.loc[dados["inflacao_anual_pct"] >= 10, "faixa_inflacao"] = "10% ou mais"

dados[["ano", "pais", "inflacao_anual_pct", "faixa_inflacao"]].head(10)

,ano,pais,inflacao_anual_pct,faixa_inflacao
0,2020,Afghanistan,5.601888,Abaixo de 10%
1,2020,Albania,1.620887,Abaixo de 10%
2,2020,Algeria,2.415131,Abaixo de 10%
3,2020,American Samoa,NaN,Sem informação
4,2020,Andorra,NaN,Sem informação
5,2020,Angola,22.271539,10% ou mais
6,2020,Antigua and Barbuda,0.625990,Abaixo de 10%
7,2020,Argentina,42.015095,10% ou mais
8,2020,Armenia,1.211436,Abaixo de 10%
9,2020,Aruba,NaN,Sem informação


### 4.1 Agregação condicional: somente países com inflação elevada

### Exercício — Inflação e desemprego por região

**Pergunta:** Como varia a taxa média de desemprego entre países com diferentes níveis de inflação, considerando as regiões do mundo em 2024?

Para responder, vamos classificar os países em quatro faixas de inflação:

| Faixa | Intervalo de inflação |
|---|---|
| 1 | Inflação menor que 2% |
| 2 | Inflação de 2% até menos de 5% |
| 3 | Inflação de 5% até 10% (inclusive) |
| 4 | Inflação maior que 10% |

Em seguida, vamos calcular a **taxa média de desemprego** e o **número de países** em cada faixa de inflação, separadamente por região, utilizando `groupby()` e `agg()`.


In [ ]:
# 1. Selecionar os dados de 2024 com inflação e desemprego disponíveis
dados_2024 = dados.loc[
    (dados["ano"] == 2024) &
    (dados["inflacao_anual_pct"].notna()) &
    (dados["desemprego_pct"].notna())
].copy()


# 2. Criar uma coluna para classificar a inflação
dados_2024["faixa_inflacao"] = ""

dados_2024.loc[
    dados_2024["inflacao_anual_pct"] < 2,
    "faixa_inflacao"
] = "1. Menor que 2%"

dados_2024.loc[
    (dados_2024["inflacao_anual_pct"] >= 2) &
    (dados_2024["inflacao_anual_pct"] < 5),
    "faixa_inflacao"
] = "2. De 2% a menos de 5%"

dados_2024.loc[
    (dados_2024["inflacao_anual_pct"] >= 5) &
    (dados_2024["inflacao_anual_pct"] <= 10),
    "faixa_inflacao"
] = "3. De 5% até 10%"

dados_2024.loc[
    dados_2024["inflacao_anual_pct"] > 10,
    "faixa_inflacao"
] = "4. Maior que 10%"


# 3. Agrupar por faixa de inflação
resumo_condicional = (
    dados_2024.groupby("faixa_inflacao", as_index=False)
    .agg(
        desemprego_medio_pct=("desemprego_pct", "mean"),
        numero_paises=("pais", "count")
    )
)

# 4. Ordenar e arredondar os resultados
resumo_condicional = (
    resumo_condicional
    .sort_values("faixa_inflacao")
    .round(2)
    .reset_index(drop=True)
)

display(resumo_condicional)

,faixa_inflacao,desemprego_medio_pct,numero_paises
0,1. Menor que 2%,6.15,46
1,2. De 2% a menos de 5%,6.89,81
2,3. De 5% até 10%,5.19,18
3,4. Maior que 10%,6.17,19


### 4.2 Percentual de países que atendem a uma condição

Uma coluna booleana tem valores `True`/`False`. Ao calcular `.mean()`, `True` conta como 1 e `False` como 0. A média × 100 fornece o **percentual de observações que atendem ao critério**.

**Cuidado com o denominador:** retiramos as observações com inflação ausente antes de calcular o percentual. Caso contrário, países sem informação seriam tratados como se estivessem abaixo de 10%.

In [ ]:
# True = inflação >= 10%; False = inflação informada, mas inferior a 10%.
dados["inflacao_10_ou_mais"] = dados["inflacao_anual_pct"] >= 10

# Apenas países com informação de inflação em 2024.
com_inflacao_2024 = dados.loc[
    (dados["ano"] == 2024) & dados["inflacao_anual_pct"].notna()
]

percentual_alta = (
    com_inflacao_2024.groupby("regiao", as_index=False)
    .agg(
        fracao_de_paises=("inflacao_10_ou_mais", "mean"),
        paises_com_inflacao=("pais", "count")
    )
)
percentual_alta["percentual_paises_pct"] = 100 * percentual_alta["fracao_de_paises"]

percentual_alta[["regiao", "paises_com_inflacao", "percentual_paises_pct"]]

#### Exercício 4 — operações condicionais

1. Crie `faixa_desemprego` com três categorias: **"Sem informação"**, **"Até 8%"** e **"Acima de 8%"**. Use `.loc[]` e não classifique ausências como desemprego baixo.
2. Em 2024, calcule a **inflação média por região** apenas para países com **desemprego superior a 8%** e inflação informada.
3. Em 2024, calcule o **percentual de países com desemprego acima de 8% por região**, considerando no denominador somente países com desemprego informado.
4. Explique a diferença entre o **número de países selecionados** e o **percentual de países selecionados**.

In [ ]:
# EXERCÍCIO 4 — complete aqui.
# Dicas: atribuição condicional com .loc[], filtragem, .agg() e média de booleanos.

---
## 5. Exercício final — um diagnóstico regional em 2024

**Pergunta:** quais regiões apresentaram maior **PIB nominal per capita agregado** entre os países com dados disponíveis, e como se comparam em inflação e desemprego?

Construa uma tabela a partir de `dados`:

1. Filtre somente **2024**.
2. Agrupe por **região** e calcule: soma do PIB (`pib_usd`), soma da população (`populacao_total`), média da inflação, média do desemprego e número de países com inflação informada.
3. Crie `pib_per_capita_agregado_usd = pib_total_usd / populacao_total`.
4. Ordene por `pib_per_capita_agregado_usd` em ordem decrescente.
5. Interprete a diferença entre **razão dos totais regionais** e **média simples dos PIBs per capita dos países**.

**Cuidado metodológico:** se PIB ou população estiverem ausentes para alguns países, os totais podem ter coberturas diferentes. A razão calculada só representa a região corretamente quando as coberturas são compatíveis. Ela não é uma estimativa oficial de PIB per capita regional.

In [ ]:
# Exercício final — complete aqui.
# 1. Filtrar 2024.
# 2. Agrupar por região e calcular totais/médias/contagens.
# 3. Dividir PIB total pela população total.
# 4. Ordenar e interpretar o resultado.

---
## Síntese

| Pergunta | Recurso do Pandas |
|---|---|
| Qual o total, média ou contagem? | `.sum()`, `.mean()`, `.count()`, `.size()` |
| Como comparar regiões? | `.groupby("regiao")` |
| Como comparar regiões em cada ano? | `.groupby(["ano", "regiao"])` |
| Como calcular estatísticas diferentes em grupos? | `.agg()` |
| Como criar variáveis? | `dados["nova"] = ...` |
| Como repetir um total por grupo em cada linha? | `.groupby(...).transform("sum")` |
| Como calcular participações? | Valor individual ÷ total do grupo × 100 |
| Como filtrar linhas? | `.loc[condicao, colunas]` |
| Como combinar condições? | `&`, `\|`, `~`, `.isin()` |
| Como lidar com ausências? | `.isna()`, `.notna()` |
| Como classificar sem perder linhas? | Atribuições condicionais com `.loc[]` |
| Como contar proporção que atende a uma regra? | Média de booleanos, **somente no conjunto válido** |



---
# Gabarito dos exercícios — respostas recolhidas

Clique em **Mostrar solução** para consultar cada resposta. Os códigos podem ser copiados para uma célula de código e executados no Colab. Execute primeiro as células que constroem o DataFrame `dados`.

**Observação:** as médias de inflação e desemprego são médias simples entre países, e o PIB per capita regional requer que PIB e população correspondam ao mesmo conjunto de países.

In [ ]:
<details>
<summary><b>Mostrar solução — Exercício 1 — Agregações</b></summary>

```python
# 1. População total dos países com informação em 2024
paises_2024 = dados.loc[dados["ano"] == 2024]

populacao_total_2024 = paises_2024.loc[
    paises_2024["populacao_total"].notna(), "populacao_total"
].sum()

print("População total em 2024:", f"{populacao_total_2024:,.0f}")

# 2. Desemprego médio por região em 2024
media_desemprego_regiao = (
    paises_2024.groupby("regiao", as_index=False)
    .agg(desemprego_medio_pct=("desemprego_pct", "mean"))
    .sort_values("desemprego_medio_pct", ascending=False)
    .round(2)
    .reset_index(drop=True)
)
display(media_desemprego_regiao)

# 3. Estatísticas por ano e região
regiao_ano_ex1 = (
    dados.groupby(["ano", "regiao"], as_index=False)
    .agg(
        populacao_total=("populacao_total", "sum"),
        inflacao_media_pct=("inflacao_anual_pct", "mean"),
        observacoes_com_desemprego=("desemprego_pct", "count")
    )
    .sort_values(["ano", "regiao"])
    .reset_index(drop=True)
)
display(regiao_ano_ex1)
```

</details>

In [ ]:
# 1. Repetir em cada linha a população da região no mesmo ano
dados["populacao_regiao_ano"] = (
    dados.groupby(["ano", "regiao"])["populacao_total"]
    .transform("sum")
)

# 2. Participação da população do país na região (%)
dados["participacao_pop_regional_pct"] = (
    100 * dados["populacao_total"] / dados["populacao_regiao_ano"]
)

display(dados[[
    "ano", "pais", "regiao", "populacao_total",
    "populacao_regiao_ano", "participacao_pop_regional_pct"
]].head(10))

# 3. Conferir se a soma das participações se aproxima de 100%
verificacao_populacao = (
    dados.groupby(["ano", "regiao"], as_index=False)
    .agg(soma_participacoes_pct=("participacao_pop_regional_pct", "sum"))
    .round(2)
)
display(verificacao_populacao)

# 4. Interpretação:
# O PIB per capita é uma razão (PIB / população), não um total.
# Somar PIBs per capita de países não produz o PIB per capita regional.
# Para o PIB per capita agregado, somamos primeiro os PIBs e as
# populações dos MESMOS países e depois dividimos os totais.

In [ ]:
# 1. América Latina e Caribe em 2024, com desemprego informado
america_latina_2024 = dados.loc[
    (dados["ano"] == 2024) &
    (dados["regiao"] == "Latin America & Caribbean") &
    (dados["desemprego_pct"].notna())
]
print("1. Países selecionados:", america_latina_2024.shape[0])
display(america_latina_2024)

# 2. Inflação > 8% OU desemprego > 12%, em 2024
inflacao_ou_desemprego = dados.loc[
    (dados["ano"] == 2024) &
    ((dados["inflacao_anual_pct"] > 8) |
     (dados["desemprego_pct"] > 12))
]
print("2. Países selecionados:", inflacao_ou_desemprego.shape[0])
display(inflacao_ou_desemprego)

# 3. PIB ausente em 2024
pib_ausente_2024 = dados.loc[
    (dados["ano"] == 2024) & dados["pib_usd"].isna()
]
print("3. Observações com PIB ausente:", pib_ausente_2024.shape[0])
display(pib_ausente_2024.head(10))

# 4. Seleção de 2020 e 2021 e seu complemento
condicao_2020_2021 = dados["ano"].isin([2020, 2021])
anos_2020_2021 = dados.loc[condicao_2020_2021]
complemento = dados.loc[~condicao_2020_2021]

print("4. Anos 2020 e 2021:", anos_2020_2021.shape[0])
print("4. Complemento (outros anos):", complemento.shape[0])
display(anos_2020_2021.head(10))
display(complemento.head(10))

In [ ]:
# 1. Classificar o desemprego, sem confundir ausências com valores baixos
dados["faixa_desemprego"] = "Sem informação"

dados.loc[
    dados["desemprego_pct"].notna() & (dados["desemprego_pct"] <= 8),
    "faixa_desemprego"
] = "Até 8%"

dados.loc[
    dados["desemprego_pct"] > 8,
    "faixa_desemprego"
] = "Acima de 8%"

display(dados[["pais", "ano", "desemprego_pct", "faixa_desemprego"]].head(10))

# 2. Inflação média regional entre países com desemprego > 8%
casos_desemprego_alto = dados.loc[
    (dados["ano"] == 2024) &
    (dados["desemprego_pct"] > 8) &
    dados["inflacao_anual_pct"].notna()
]

inflacao_condicional_regiao = (
    casos_desemprego_alto.groupby("regiao", as_index=False)
    .agg(
        inflacao_media_pct=("inflacao_anual_pct", "mean"),
        numero_paises=("pais", "count")
    )
    .round(2)
)
display(inflacao_condicional_regiao)

# 3. Percentual de países com desemprego > 8%, por região
# Denominador: apenas países com desemprego informado em 2024.
desemprego_informado_2024 = dados.loc[
    (dados["ano"] == 2024) & dados["desemprego_pct"].notna()
].copy()

desemprego_informado_2024["acima_8"] = (
    desemprego_informado_2024["desemprego_pct"] > 8
)

percentual_regional = (
    desemprego_informado_2024.groupby("regiao", as_index=False)
    .agg(
        numero_paises=("pais", "count"),
        proporcao_acima_8=("acima_8", "mean")
    )
)
percentual_regional["percentual_acima_8_pct"] = (
    100 * percentual_regional["proporcao_acima_8"]
)
display(percentual_regional[[
    "regiao", "numero_paises", "percentual_acima_8_pct"
]].round(2))

# 4. Interpretação:
# Número de países = quantidade absoluta que satisfaz a condição.
# Percentual = número que satisfaz a condição / número com dado
# de desemprego disponível na região, multiplicado por 100.
# Aqui cada país tem o mesmo peso, independentemente do tamanho.

In [ ]:
# 1. Selecionar 2024 e manter países com PIB e população disponíveis
# Assim, PIB total e população total têm a MESMA cobertura de países.
base_integrada = dados.loc[
    (dados["ano"] == 2024) &
    dados["pib_usd"].notna() &
    dados["populacao_total"].notna() &
    (dados["populacao_total"] > 0)
].copy()

# 2. Agregar por região
resumo_integrado = (
    base_integrada.groupby("regiao", as_index=False)
    .agg(
        pib_total_usd=("pib_usd", "sum"),
        populacao_total=("populacao_total", "sum"),
        inflacao_media_pct=("inflacao_anual_pct", "mean"),
        desemprego_medio_pct=("desemprego_pct", "mean"),
        paises_com_inflacao=("inflacao_anual_pct", "count")
    )
)

# 3. PIB nominal per capita agregado (razão dos totais)
resumo_integrado["pib_per_capita_agregado_usd"] = (
    resumo_integrado["pib_total_usd"] / resumo_integrado["populacao_total"]
)

# 4. Ordenar e exibir os resultados
resumo_integrado = (
    resumo_integrado.sort_values(
        "pib_per_capita_agregado_usd", ascending=False
    )
    .round(2)
    .reset_index(drop=True)
)
display(resumo_integrado)

# 5. Interpretação:
# Razão dos totais = soma dos PIBs / soma das populações dos
# países incluídos; equivale a uma média ponderada pela população.
# Média simples = média dos PIBs per capita nacionais, dando
# o mesmo peso a países grandes e pequenos.
# As médias de inflação e desemprego aqui também se referem
# aos países incluídos em base_integrada e com essas taxas disponíveis.